# Búsqueda cuántica con Grover

Este cuaderno demuestra el uso del **algoritmo de Grover** para encontrar un elemento en una lista no estructurada de un millón de datos. El código utiliza la librería [Qiskit](https://qiskit.org/) y puede ejecutarse localmente en una máquina con capacidad de simular circuitos de 20 qubits.

## Instalación

Si aún no tienes las dependencias instaladas, puedes crear el entorno definido en `environment.yml` o instalar los paquetes básicos con pip:

```bash
pip install qiskit qiskit-aer jupyterlab matplotlib
```

In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
import math

# Configuración del problema
n_qubits = 20                     # 2^20 = 1 048 576 elementos
N = 2 ** n_qubits
objetivo = '10101010101010101010'  # Estado marcado que deseamos encontrar

print(f'Tamaño de la base de datos: {N}')
print(f'Elemento objetivo: {objetivo}')

Tamaño de la base de datos: 1048576
Elemento objetivo: 10101010101010101010


## Construcción del oráculo y del operador de difusión

El oráculo marca el estado objetivo aplicando un giro de fase. El operador de difusión invierte las amplitudes respecto a la media para amplificar la probabilidad de medir el estado marcado.

In [2]:
# Oráculo para el estado objetivo
oracle = QuantumCircuit(n_qubits)
for i, bit in enumerate(objetivo):
    if bit == '0':
        oracle.x(i)
oracle.h(n_qubits - 1)
oracle.mcx(list(range(n_qubits - 1)), n_qubits - 1)
oracle.h(n_qubits - 1)
for i, bit in enumerate(objetivo):
    if bit == '0':
        oracle.x(i)

# Operador de difusión (Grover diffusion)
diffusion = QuantumCircuit(n_qubits)
diffusion.h(range(n_qubits))
diffusion.x(range(n_qubits))
diffusion.h(n_qubits - 1)
diffusion.mcx(list(range(n_qubits - 1)), n_qubits - 1)
diffusion.h(n_qubits - 1)
diffusion.x(range(n_qubits))
diffusion.h(range(n_qubits))

## Ejecución del algoritmo

El número óptimo de iteraciones es aproximadamente $\pi/4\sqrt{N}$. Para $N = 2^{20}$ esto equivale a 804 iteraciones. Ejecutamos el circuito en el simulador de Qiskit.

In [3]:
iteraciones = int(math.pi/4 * math.sqrt(N))
print(f'Iteraciones de Grover: {iteraciones}')

qc = QuantumCircuit(n_qubits)
qc.h(range(n_qubits))
for _ in range(iteraciones):
    qc.compose(oracle, inplace=True)
    qc.compose(diffusion, inplace=True)
qc.measure_all()

backend = Aer.get_backend('aer_simulator')
job = backend.run(transpile(qc, backend), shots=1)
resultado = job.result().get_counts()
print(resultado)

Iteraciones de Grover: 804


{'01010101010101010101': 1}


## Interpretación del resultado

La clave devuelta por `get_counts` corresponde al elemento encontrado. Con alta probabilidad, el algoritmo de Grover nos da el estado objetivo en una única medida.